# Sorare Player Score Analysis

Run this after the extraction notebook has produced `data_preparation/datasets/sorare/all_players_scores_long.csv`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from load_data import load_data
from inspect_dataset import inspect_dataset
from clean_data import clean_scores
from feature_engineering import add_features
from analyse_scores import build_rankings, write_summary
from visualise import save_charts
from model_predictions import train_baseline_model

OUTPUT_DIR = PROJECT_ROOT / "analyses" / "sorare_scores"
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "charts").mkdir(exist_ok=True)

In [ ]:
input_path = PROJECT_ROOT / "data_preparation" / "datasets" / "sorare" / "all_players_scores_long.csv"
raw = load_data(input_path)
inspect_report = inspect_dataset(raw)

In [ ]:
cleaned, reports = clean_scores(raw)
cleaned.to_csv(OUTPUT_DIR / "cleaned_sorare_scores.csv", index=False)
for name, report in reports.items():
    if not report.empty:
        report.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
cleaned.head()

In [ ]:
engineered = add_features(cleaned)
engineered.to_csv(OUTPUT_DIR / "engineered_sorare_scores.csv", index=False)
engineered.head()

In [ ]:
tables = build_rankings(engineered)
tables["player_rankings"].to_csv(OUTPUT_DIR / "player_rankings.csv", index=False)
tables["best_value_players"].to_csv(OUTPUT_DIR / "value_rankings.csv", index=False)
write_summary(OUTPUT_DIR / "analysis_summary.md", tables)

for name in ["most_consistent_players", "highest_average_scores", "highest_ceiling_players", "improving_players", "declining_players", "high_risk_high_reward_players"]:
    print("\n", name)
    display(tables[name].head(10))

In [ ]:
save_charts(engineered, OUTPUT_DIR / "charts")
print("Charts saved to", OUTPUT_DIR / "charts")

In [ ]:
prediction_results, metrics = train_baseline_model(engineered)
prediction_results.to_csv(OUTPUT_DIR / "prediction_results.csv", index=False)
metrics